# RecallAI — Personal RAG Engine
### Completed AI/ML University Project Prototype

This notebook implements the core **Personal RAG** pipeline described in the supplied skeleton:

**Capture → Understand → Store → Retrieve → Rerank → Generate → Cite**

The prototype uses synthetic personal-memory data. It demonstrates preprocessing, chunking, TF-IDF retrieval, metadata filtering, hybrid retrieval, heuristic reranking, grounded answer generation, multilingual/Banglish query testing, and retrieval evaluation.

> **Privacy:** No real WhatsApp, email, browser, or personal-account data is accessed.


## 1. Project Objective

The problem is that useful personal information can be scattered across notes, PDFs, screenshots, voice notes, webpages, and messages.

RecallAI converts these sources into searchable memory and answers questions using retrieved personal context.

### Main objectives
1. Normalize and process memory.
2. Split long content into retrieval chunks.
3. Create vector representations.
4. Retrieve relevant memories.
5. Apply metadata and keyword signals.
6. Rerank the best candidates.
7. Generate an answer grounded in retrieved evidence.
8. Show source IDs for explainability.
9. Evaluate retrieval using Recall@K and MRR.
10. Support English, Bangla, and Banglish query experiments.

The supplied skeleton specifies transformer embeddings, vector databases, OCR/ASR, and an actual LLM as production extensions. This notebook keeps the core prototype dependency-light and executable.


## 2. System Architecture

```text
Notes / PDFs / Web / Screenshots / Voice
                 ↓
          Ingestion Layer
                 ↓
       Cleaning + Chunking
                 ↓
          Representation
       TF-IDF baseline here
                 ↓
      Semantic + Keyword Search
                 ↓
             Reranking
                 ↓
         Retrieved Context
                 ↓
       Grounded RAG Response
                 ↓
       Answer + Source IDs
```

Production extensions from the supplied design can replace the baseline with multilingual transformer embeddings, pgvector/Qdrant, OCR, Whisper/ASR, and an LLM.


## 3. Environment Setup

In [ ]:
import re
import random
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Environment ready.")


## 4. Synthetic Personal Memory Dataset

Each memory has the fields required by the supplied skeleton:
- memory_id
- source_type
- title
- content
- date
- category
- language
- url
- importance


In [ ]:
memory_data = [
    ["mem_001", "note", "AI Lab Dataset",
     "For the machine learning classification task, we decided to use the heart disease dataset.",
     "2026-09-12", "academic", "English", "", 0.90],

    ["mem_002", "voice", "TrustWheels Task",
     "Prepare the financial, technical, and HR feasibility section for the TrustWheels SAD presentation.",
     "2026-09-20", "academic", "English", "", 0.95],

    ["mem_003", "webpage", "RAG Article",
     "Retrieval augmented generation improves factual responses by retrieving relevant context before generation.",
     "2026-09-25", "research", "English", "https://example.com/rag", 0.80],

    ["mem_004", "screenshot", "Amato Pricing",
     "Vanilla Perfection cake price is 999 BDT per pound.",
     "2026-09-28", "business", "English", "", 0.92],

    ["mem_005", "note", "Banglish AI Note",
     "amar AI assignment er dataset ta classification er jonno use korbo. Dataset hocche heart disease.",
     "2026-09-29", "academic", "Banglish", "", 0.84],

    ["mem_006", "pdf", "RAG Architecture",
     "The architecture contains ingestion, preprocessing, chunking, embedding, vector storage, retrieval, reranking and generation.",
     "2026-10-01", "project", "English", "", 0.96],

    ["mem_007", "voice", "Project Idea",
     "Build an omnichannel second brain that remembers notes, screenshots, browser resources and audio.",
     "2026-10-02", "project", "English", "", 0.98],

    ["mem_008", "webpage", "Multilingual Embeddings",
     "Multilingual embedding models can map Bangla, English and mixed-language queries into a shared semantic space.",
     "2026-10-03", "research", "English", "https://example.com/embeddings", 0.88],

    ["mem_009", "note", "CSE Study Plan",
     "Review data structures, object oriented programming and database concepts before the next CSE assessment.",
     "2026-10-03", "academic", "English", "", 0.82],

    ["mem_010", "screenshot", "Project Deadline",
     "The final project presentation is scheduled for the second week of October.",
     "2026-10-04", "project", "English", "", 0.91],
]

columns = ["memory_id","source_type","title","content","date",
           "category","language","url","importance"]

memories = pd.DataFrame(memory_data, columns=columns)
print("Total memories:", len(memories))
display(memories)


## 5. Data Profiling

In [ ]:
memories["content_length"] = memories["content"].str.len()

profile = pd.DataFrame({
    "metric": ["total memories", "missing content", "unique source types",
               "unique categories", "unique languages"],
    "value": [
        len(memories),
        int(memories["content"].isna().sum()),
        memories["source_type"].nunique(),
        memories["category"].nunique(),
        memories["language"].nunique(),
    ]
})
display(profile)

print("Source distribution")
display(memories["source_type"].value_counts().to_frame("count"))

print("Category distribution")
display(memories["category"].value_counts().to_frame("count"))


## 6. Preprocessing and Chunking

Whitespace is normalized first. Long content is divided into overlapping word chunks so that retrieval can return focused evidence rather than an entire document.


In [ ]:
def normalize_text(text):
    return re.sub(r"\s+", " ", str(text)).strip()

def chunk_text(text, chunk_size=50, overlap=10):
    words = normalize_text(text).split()
    if not words:
        return []
    if len(words) <= chunk_size:
        return [" ".join(words)]

    chunks = []
    start = 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunks.append(" ".join(words[start:end]))
        if end == len(words):
            break
        start = max(end - overlap, start + 1)
    return chunks

memories["clean_text"] = memories["content"].apply(normalize_text)

records = []
for _, row in memories.iterrows():
    for i, chunk in enumerate(chunk_text(row["clean_text"])):
        records.append({
            "memory_id": row["memory_id"],
            "chunk_id": f"{row['memory_id']}_chunk_{i}",
            "source_type": row["source_type"],
            "category": row["category"],
            "language": row["language"],
            "date": row["date"],
            "importance": row["importance"],
            "title": row["title"],
            "text": chunk,
        })

chunks_df = pd.DataFrame(records)
print("Total chunks:", len(chunks_df))
display(chunks_df.head(15))


## 7. TF-IDF Baseline Representation

The supplied skeleton explicitly uses TF-IDF as a dependency-light baseline and recommends comparing it with multilingual transformer embeddings later.

TF-IDF is not a neural embedding model; it is used here to make the prototype reproducible without downloading a large model.


In [ ]:
tfidf = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=1,
    sublinear_tf=True
)

tfidf_matrix = tfidf.fit_transform(chunks_df["text"].tolist())
print("TF-IDF matrix shape:", tfidf_matrix.shape)


## 8. Semantic Retrieval

In [ ]:
def semantic_search(query, top_k=5, dataframe=None, matrix=None):
    dataframe = chunks_df if dataframe is None else dataframe
    matrix = tfidf_matrix if matrix is None else matrix

    query = normalize_text(query)
    if not query:
        return dataframe.iloc[0:0].copy()

    q_vec = tfidf.transform([query])
    scores = cosine_similarity(q_vec, matrix).ravel()

    result = dataframe.copy()
    result["semantic_score"] = scores
    return result.sort_values(
        "semantic_score", ascending=False
    ).head(top_k).reset_index(drop=True)

demo_query = "Where did I save the dataset for my AI classification assignment?"
display(semantic_search(demo_query)[
    ["memory_id","title","text","semantic_score"]
])


## 9. Metadata Filtering

Personal memory retrieval should support structured constraints such as category, source type, language, date, and importance.


In [ ]:
def filter_memories(df, category=None, source_type=None, language=None,
                     min_importance=None, start_date=None, end_date=None):
    result = df.copy()

    if category:
        result = result[result["category"].eq(category)]
    if source_type:
        result = result[result["source_type"].eq(source_type)]
    if language:
        result = result[result["language"].eq(language)]
    if min_importance is not None:
        result = result[result["importance"] >= min_importance]
    if start_date:
        result = result[result["date"] >= start_date]
    if end_date:
        result = result[result["date"] <= end_date]

    return result.reset_index(drop=True)

display(filter_memories(
    chunks_df, category="project", min_importance=0.90
)[["memory_id","date","importance","text"]])


## 10. Hybrid Retrieval

Semantic similarity is useful for paraphrases. Exact keyword matching is useful for names, prices, IDs, and dates.

The prototype combines the semantic score with a lightweight term-overlap signal.


In [ ]:
def keyword_overlap_score(query, text):
    query_terms = set(re.findall(r"\w+", normalize_text(query).lower()))
    text_terms = set(re.findall(r"\w+", normalize_text(text).lower()))

    if not query_terms:
        return 0.0

    overlap = len(query_terms & text_terms) / len(query_terms)
    return min(overlap, 1.0)

def hybrid_search(query, top_k=5, alpha=0.70, dataframe=None):
    dataframe = chunks_df if dataframe is None else dataframe
    if dataframe.empty:
        return dataframe.copy()

    local_matrix = tfidf.transform(dataframe["text"].tolist())
    q_vec = tfidf.transform([normalize_text(query)])
    semantic = cosine_similarity(q_vec, local_matrix).ravel()

    result = dataframe.copy()
    result["semantic_score"] = semantic
    result["keyword_score"] = [
        keyword_overlap_score(query, text)
        for text in result["text"]
    ]

    result["hybrid_score"] = (
        alpha * result["semantic_score"]
        + (1 - alpha) * result["keyword_score"]
    )

    return result.sort_values(
        "hybrid_score", ascending=False
    ).head(top_k).reset_index(drop=True)

display(hybrid_search(demo_query, top_k=5)[
    ["memory_id","text","semantic_score","keyword_score","hybrid_score"]
])


## 11. Reranking

A production implementation can use a cross-encoder such as a BGE-style reranker. For this executable prototype, reranking combines retrieval relevance with memory importance.

This is intentionally a transparent heuristic rather than a claim that a cross-encoder was trained here.


In [ ]:
def rerank_candidates(query, candidates, top_k=5):
    if candidates.empty:
        return candidates.copy()

    result = candidates.copy()

    result["rerank_score"] = (
        0.80 * result["hybrid_score"]
        + 0.20 * result["importance"]
    )

    return result.sort_values(
        "rerank_score", ascending=False
    ).head(top_k).reset_index(drop=True)

candidates = hybrid_search(demo_query, top_k=8)
reranked = rerank_candidates(demo_query, candidates, top_k=5)

display(reranked[
    ["memory_id","title","text","hybrid_score","importance","rerank_score"]
])


## 12. Grounded Personal RAG

The grounding rule is:

**No reliable evidence → No factual claim**

The prototype answer generator uses the highest-ranked evidence and always returns source IDs. An actual LLM can later consume the same context.


In [ ]:
def generate_grounded_answer(query, results, min_score=0.08, source_margin=0.12):
    """Generate a grounded prototype answer and return only supporting sources.

    The top-ranked result is treated as the primary evidence.
    Additional sources are included only when they are close enough to the
    primary score to plausibly support the same answer.
    """
    if results.empty:
        return {
            "answer": "I could not find a reliable memory for this question.",
            "sources": [],
            "confidence": 0.0,
            "evidence": []
        }

    usable = results[results["rerank_score"] >= min_score].copy()
    if usable.empty:
        return {
            "answer": "I could not find a reliable memory for this question.",
            "sources": [],
            "confidence": 0.0,
            "evidence": []
        }

    usable = usable.sort_values("rerank_score", ascending=False).reset_index(drop=True)
    top = usable.iloc[0]

    # Keep only sources that are close to the strongest evidence.
    supporting = usable[
        usable["rerank_score"] >= max(min_score, float(top["rerank_score"]) - source_margin)
    ].copy()

    # Remove duplicate memory IDs while preserving ranking.
    supporting = supporting.drop_duplicates("memory_id")

    answer = (
        f"Based on my stored memory, {top['text']} "
        f"(source: {top['memory_id']})."
    )

    return {
        "answer": answer,
        "sources": supporting["memory_id"].tolist(),
        "confidence": round(float(top["rerank_score"]), 3),
        "evidence": supporting["text"].tolist()
    }


## 13. End-to-End Query Function

This function runs the complete prototype pipeline:

**Query → Hybrid Retrieval → Reranking → Grounded Answer → Sources**


In [ ]:
def ask_recall_ai(query, top_k=8, final_k=5):
    if not normalize_text(query):
        return {
            "answer": "Please enter a question.",
            "sources": [],
            "confidence": 0.0,
            "retrieved": pd.DataFrame()
        }

    candidates = hybrid_search(query, top_k=top_k)
    final_results = rerank_candidates(query, candidates, top_k=final_k)
    response = generate_grounded_answer(query, final_results)

    response["retrieved"] = final_results
    return response

response = ask_recall_ai(
    "Where did I save my AI assignment dataset?"
)

print("ANSWER:", response["answer"])
print("SOURCES:", response["sources"])
print("CONFIDENCE:", response["confidence"])
display(response["retrieved"][
    ["memory_id","title","text","rerank_score"]
])


## 14. Multilingual / Banglish Retrieval Experiment

The supplied project design requires English, Bangla, and Banglish support. TF-IDF is only a baseline and will not provide strong cross-language semantic matching by itself.

Therefore, this experiment demonstrates the query set while making the limitation explicit. A production system should use a multilingual transformer embedding model.


In [ ]:
test_queries = [
    "Where did I save my AI assignment dataset?",
    "আমি আমার AI assignment-এর dataset কোথায় save করেছিলাম?",
    "amar AI assignment er dataset ta kothay save korsilam?"
]

for q in test_queries:
    print("\nQUERY:", q)
    result = hybrid_search(q, top_k=3)
    display(result[[
        "memory_id","title","language","text","hybrid_score"
    ]])


## 15. Adding a New Memory

This simulates the ingestion layer for a new text note.


In [ ]:
def ingest_text_memory(content, source_type="note", title="Untitled",
                        category="general", language="English",
                        importance=0.50, date=None):
    return {
        "memory_id": f"mem_new_{len(memories)+1:03d}",
        "source_type": source_type,
        "title": title,
        "content": normalize_text(content),
        "date": date or datetime.now().strftime("%Y-%m-%d"),
        "category": category,
        "language": language,
        "url": "",
        "importance": float(importance)
    }

new_memory = ingest_text_memory(
    "Compare multilingual embedding models for Bangla and Banglish retrieval.",
    title="Embedding Experiment",
    category="research",
    importance=0.88
)

print(new_memory)


## 16. Automatic Memory Intelligence Schema

A production AI service can attach category, topics, entities, summary, importance, actions, and dates to each memory. These become searchable metadata.


In [ ]:
memory_intelligence_schema = {
    "category": "research",
    "topics": ["RAG", "embeddings", "multilingual retrieval"],
    "entities": [
        {"name": "RecallAI", "type": "project"}
    ],
    "summary": "Experiment with multilingual retrieval for Bangla and Banglish queries.",
    "importance_score": 0.88,
    "action_items": ["Compare embedding models"],
    "dates": []
}

display(pd.DataFrame([memory_intelligence_schema]))


## 17. Evaluation Dataset

Each evaluation item contains:
- query
- expected memory IDs
- language
- difficulty

The questions include semantic, multilingual, business, project, and architecture examples.


In [ ]:
evaluation_data = pd.DataFrame([
    ["Which dataset did I choose for my classification assignment?",
     ["mem_001"], "English", "easy"],

    ["amar AI assignment er dataset ta ki chilo?",
     ["mem_001", "mem_005"], "Banglish", "medium"],

    ["What was my TrustWheels presentation responsibility?",
     ["mem_002"], "English", "easy"],

    ["What price did I set for the vanilla cake?",
     ["mem_004"], "English", "easy"],

    ["What components are in my personal RAG architecture?",
     ["mem_006"], "English", "medium"],

    ["What is the main idea of my second brain project?",
     ["mem_007"], "English", "medium"],
], columns=["query","expected_memory_ids","language","difficulty"])

display(evaluation_data)


## 18. Retrieval Evaluation

### Recall@K
Measures how many expected memories appear in the top K results.

### MRR
Measures how high the first relevant memory appears.

For a personal memory system, both are important because the correct memory should be found and ranked near the top.


In [ ]:
def recall_at_k(results, expected_ids, k=5):
    retrieved = set(results.head(k)["memory_id"])
    expected = set(expected_ids)
    return len(retrieved & expected) / max(len(expected), 1)

def reciprocal_rank(results, expected_ids):
    expected = set(expected_ids)

    for rank, memory_id in enumerate(results["memory_id"], start=1):
        if memory_id in expected:
            return 1.0 / rank
    return 0.0

rows = []

for _, row in evaluation_data.iterrows():
    result = hybrid_search(row["query"], top_k=5)
    rows.append({
        "query": row["query"],
        "language": row["language"],
        "difficulty": row["difficulty"],
        "recall@5": recall_at_k(result, row["expected_memory_ids"], 5),
        "MRR": reciprocal_rank(result, row["expected_memory_ids"])
    })

evaluation_results = pd.DataFrame(rows)
display(evaluation_results)

print("Mean Recall@5:", round(evaluation_results["recall@5"].mean(), 3))
print("Mean MRR:", round(evaluation_results["MRR"].mean(), 3))


## 19. Evaluation Visualization

In [ ]:
metrics = {
    "Recall@5": evaluation_results["recall@5"].mean(),
    "MRR": evaluation_results["MRR"].mean()
}

plt.figure(figsize=(6,4))
plt.bar(metrics.keys(), metrics.values())
plt.ylim(0, 1.05)
plt.ylabel("Score")
plt.title("RecallAI Retrieval Performance")
plt.tight_layout()
plt.show()


## 20. Source-Type Analysis

In [ ]:
source_counts = memories["source_type"].value_counts()

plt.figure(figsize=(7,4))
plt.bar(source_counts.index, source_counts.values)
plt.ylabel("Number of memories")
plt.xlabel("Source type")
plt.title("Synthetic Memory Collection by Source Type")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


## 21. Model Comparison Plan

The supplied project specification recommends comparing:

1. TF-IDF baseline
2. Transformer embedding + vector search
3. Transformer embedding + hybrid retrieval
4. Transformer embedding + hybrid retrieval + reranker

The final report should contain **measured** Recall@5, MRR, and latency. Results must not be invented.

| System | Recall@5 | MRR | Latency |
|---|---:|---:|---:|
| TF-IDF | measured by this notebook | measured by this notebook | measure |
| Transformer | future experiment | future experiment | measure |
| Hybrid | future experiment | future experiment | measure |
| Hybrid + Reranker | future experiment | future experiment | measure |


## 22. Production Vector Database Schema

The supplied architecture recommends PostgreSQL + pgvector or Qdrant.

A memory record can contain:

```text
memory_id
user_id
chunk_id
text
source_type
category
language
date
importance
entities
embedding
source_url
```

The actual database connection belongs in a backend service.


In [ ]:
VECTOR_DB_BACKEND = "pgvector"  # production option; Qdrant is another option

memory_record_schema = {
    "memory_id": "string",
    "user_id": "string",
    "chunk_id": "string",
    "text": "string",
    "metadata": {
        "source_type": "string",
        "category": "string",
        "language": "string",
        "date": "datetime",
        "importance": "float",
        "entities": "list"
    },
    "embedding": "vector",
    "source_url": "string"
}

print("Production vector backend:", VECTOR_DB_BACKEND)
display(pd.DataFrame([memory_record_schema]))


## 23. Multimodal Ingestion Design

The production pipeline can normalize different modalities into the same memory object:

- **Text:** Cleaner → Chunker
- **PDF:** Parser → Text/table extraction → Chunker
- **Screenshot:** OCR/Vision → Text → Chunker
- **Voice:** Whisper/ASR → Transcript → Chunker
- **Web:** URL extraction → Clean text → Chunker

After normalization, every modality can share the retrieval and RAG pipeline.


## 24. Privacy and Robustness Requirements

The supplied design identifies these production requirements:

- Empty-query handling
- No-result handling
- Duplicate detection
- OCR/ASR error handling
- Conflicting memories
- Old vs. updated information
- Memory deletion
- Encryption
- User-level access control
- Per-user vector isolation
- Data export
- Consent for integrations

This notebook handles empty queries and no-result behavior at prototype level. Production security is not implemented here.


## 25. Final Demonstration

In [ ]:
demo_query = "amar AI assignment er dataset ta kothay save korsilam?"

demo_response = ask_recall_ai(demo_query)

print("USER:", demo_query)
print("\nRECALLAI ANSWER:")
print(demo_response["answer"])
print("\nSOURCE IDS:", demo_response["sources"])
print("CONFIDENCE:", demo_response["confidence"])

display(demo_response["retrieved"][
    ["memory_id","source_type","date","text","rerank_score"]
])


## 26. Algorithm Justification

**TF-IDF baseline:** provides a simple reproducible lexical representation.

**Hybrid retrieval:** combines semantic similarity with exact-term overlap, which is useful for names, prices, IDs, and dates.

**Reranking:** improves the ordering of retrieved candidates using relevance and memory importance.

**RAG:** grounds the answer in retrieved personal information instead of relying only on general model knowledge.

**Source citations:** make personal AI responses more explainable and trustworthy.

**Multilingual embeddings:** are the recommended next step for strong Bangla, English, and Banglish retrieval.


## 27. Future Work

### Phase 1 — MVP
Text notes, PDF upload, semantic search, RAG, and citations.

### Phase 2 — Multimodal
Screenshot OCR, voice transcription, image understanding, and webpage capture.

### Phase 3 — Advanced Retrieval
Multilingual embeddings, BM25 + vector search, reranking, and temporal retrieval.

### Phase 4 — Personal Intelligence
Summaries, entities, knowledge graph, related memories, and timelines.

### Phase 5 — Product
Chrome extension, mobile app, authentication, encryption, pgvector/Qdrant, and deployment.

### Phase 6 — Proactive AI
Deadline detection, reminders, related-memory suggestions, and context-aware assistance.


## 28. Conclusion

This completed prototype demonstrates the central idea of **RecallAI: a personal AI memory layer rather than a conventional note-taking application**.

The implemented research pipeline is:

```text
Synthetic Multimodal Memories
        ↓
Preprocessing + Chunking
        ↓
TF-IDF Representation
        ↓
Semantic Retrieval
        ↓
Keyword + Semantic Hybrid Retrieval
        ↓
Importance-Aware Reranking
        ↓
Grounded Personal RAG
        ↓
Answer + Source IDs
        ↓
Recall@5 + MRR Evaluation
```

The next research implementation should replace the TF-IDF baseline with multilingual transformer embeddings, connect pgvector/Qdrant, add OCR/Whisper pipelines, and connect an actual LLM while preserving the grounding and citation rules.


## 29. Quick Test Cell

Change only the question below to test RecallAI with the included synthetic memories.


In [ ]:
question = "What was my TrustWheels presentation responsibility?"
result = ask_recall_ai(question)

print("Question:", question)
print("Answer:", result["answer"])
print("Sources:", result["sources"])
print("Confidence:", result["confidence"])


## 30. Expected Clean Output

For the TrustWheels test question, the primary supporting memory should be `mem_002`. Other retrieved candidates are not automatically treated as citations.


In [ ]:
final_test = ask_recall_ai("What was my TrustWheels presentation responsibility?")

print("Question:", "What was my TrustWheels presentation responsibility?")
print("Answer:", final_test["answer"])
print("Sources:", final_test["sources"])
print("Confidence:", final_test["confidence"])
